In [1]:
import os
import json
import math
from typing import List, Dict, Any

import pandas as pd
from dotenv import load_dotenv
from tqdm import tqdm
from tenacity import retry, wait_random_exponential, stop_after_attempt

from openai import OpenAI

This notebook feeds the attachment data into the OpenAI API and generates a new free text column written in the first person that describes how the individual feels about relationships. We will use this new column to train a neural network to compare if having the additional context generates greater predictive power than using a regular classification model.

In [2]:
#check the openAI API key is loaded
load_dotenv()  
assert os.getenv("OPENAI_API_KEY"), "OPENAI_API_KEY not found. Check your .env and restart the kernel."
print("Key loaded:", os.getenv("OPENAI_API_KEY")[:6] + "…")

Key loaded: sk-pro…


In [7]:
#set up configurations
INPUT_CSV  = '/home/amybirdee/hobby_projects/attachment_style_prediction/attachment_survey.csv'
OUTPUT_CSV = '/home/amybirdee/hobby_projects/attachment_style_prediction/attachment_survey_with_text.csv'

SOURCE_CSV = OUTPUT_CSV if os.path.exists(OUTPUT_CSV) else INPUT_CSV

TEXT_COL = 'relationship_reflection'
ID_COL   = '__row_id'

BATCH_SIZE = 15                
MIN_WORDS = 90
MAX_WORDS = 150

MODEL = 'gpt-5-mini' 
MAX_OUTPUT_TOKENS = 3500        

#reads OPENAI_API_KEY from env by default
client = OpenAI()              

In [8]:
# -----------------------
# Prompt building
# -----------------------

def build_row_context(row: pd.Series) -> Dict[str, Any]:
    #Not including attachment_style_label here - we want indirect signal, not label leakage
    return {
        'age': int(row['age']) if not pd.isna(row['age']) else None,
        'gender': str(row['gender']),
        'relationship_status': str(row['relationship_status']),
        'parenting_style': str(row['parenting_style']),
        'conflict_response': str(row['conflict_response']),
        'comfort_with_intimacy': int(row['comfort_with_intimacy']),
        'fear_of_abandonment': int(row['fear_of_abandonment']),
        'trust_in_others': int(row['trust_in_others']),
        'openness_in_relationships': int(row['openness_in_relationships']),
        'social_support_perception': int(row['social_support_perception']),
        'relationship_satisfaction': int(row['relationship_satisfaction']),
    }

def build_batch_prompt(batch):
    return f"""
You are generating synthetic open-ended survey responses about romantic relationships.

For EACH person below, output exactly ONE line in this format:

<row_id>|||<generated_text>

Rules:
- Length: {MIN_WORDS}-{MAX_WORDS} words each.
- Natural, human tone. Not clinical.
- Do NOT mention attachment styles, psychology terms, or Likert scales.
- Do NOT include numerals (0–9) in the generated_text.
- Avoid repetitive phrases across responses.
- Use the attributes to shape the narrative (trust, intimacy comfort, conflict response, etc.).
- The text should be plausible, with occasional mixed signals (not perfectly aligned).
- Vary tone and phrasing across responses; avoid repeated sentence structures.
- Output ONE response per line.
- EACH line MUST start with the row_id, followed immediately by '|||'.
- Do NOT add bullet points, numbering, headings, or introductory text.
- Do NOT add blank lines.
- Do NOT add any text before or after the lines.


People:
{json.dumps(batch, ensure_ascii = False)}
""".strip()

# -----------------------
# API call (with retries)
# -----------------------

@retry(wait = wait_random_exponential(min = 1, max = 30), stop = stop_after_attempt(6))
def generate_batch(batch_payload):
    prompt = build_batch_prompt(batch_payload)

    resp = client.responses.create(
        model = MODEL,
        input = prompt,
        max_output_tokens = MAX_OUTPUT_TOKENS,
    )
    
    return resp.output_text.strip()

# -----------------------
# Main generation loop (resumable)
# -----------------------

df = pd.read_csv(SOURCE_CSV)

#add stable row ids once
if ID_COL not in df.columns:
    df.insert(0, ID_COL, range(len(df)))

#create the target column if missing
if TEXT_COL not in df.columns:
    df[TEXT_COL] = ''

#if rerunning is required, resume only missing/blank rows
missing_mask = df[TEXT_COL].isna() | (df[TEXT_COL].astype(str).str.strip().str.len() == 0)
missing_ids = df.loc[missing_mask, ID_COL].tolist()

print(f'Rows total: {len(df)}')
print(f'Rows needing generation: {len(missing_ids)}')

if len(missing_ids) == 0:
    print('Nothing to do - saving output.')
    df.to_csv(OUTPUT_CSV, index = False)
else:
    n_batches = math.ceil(len(missing_ids) / BATCH_SIZE)
    
    for b in tqdm(range(n_batches), desc = 'Generating batches'):
        batch_ids = missing_ids[b * BATCH_SIZE : (b + 1) * BATCH_SIZE]
        batch_rows = df[df[ID_COL].isin(batch_ids)].copy()

        payload = []
        for _, r in batch_rows.iterrows():
            payload.append({
                'row_id': int(r[ID_COL]),
                'attributes': build_row_context(r)})

        MAX_BATCH_RETRIES = 5
        success = False

        for attempt in range(MAX_BATCH_RETRIES):
            
            output = generate_batch(payload)

            if b == 0 and attempt == 0:
                with open('debug_batch_output.txt', 'w') as f:
                    f.write(output)

            lines = [l for l in output.splitlines() if l.strip()]
            out_map = {}

            for line in lines:
                # normalise just in case
                line = (line.replace('｜', '|').replace('\u200b', '').replace('\ufeff', ''))

                if '|||' not in line:
                    continue

                rid, text = line.split('|||', 1)
                rid = int(rid.strip())
                text = text.strip()

                if text:
                    out_map[rid] = text

            # success condition: we got ALL rows in this batch
            if len(out_map) == len(payload):
                success = True
                break

        # if still failed after retries → STOP
        if not success:
            raise RuntimeError(f'Batch {b} failed after {MAX_BATCH_RETRIES} attempts. '
            'Stopping to avoid silent data loss.')

        
        #write results back
        for rid, text in out_map.items():
            df.loc[df[ID_COL] == rid, TEXT_COL] = text

        #autosave each batch
        df.to_csv(OUTPUT_CSV, index = False)

    print('Done.')
    print('Wrote:', OUTPUT_CSV)


Rows total: 3000
Rows needing generation: 675


Generating batches: 100%|███████████████████████| 45/45 [47:36<00:00, 63.48s/it]

Done.
Wrote: /home/amybirdee/hobby_projects/attachment_style_prediction/attachment_survey_with_text.csv
